# Расчёт на проде

Загружаем готовый артефакт и применяем его к новым данным.

In [ ]:
from pathlib import Path

import pickle
import pandas as pd

from config import DATA_CSV_PART1, DATA_CSV_PART2

## Артефакт

In [ ]:
MODEL_PATH = Path('artifacts') / 'model' / 'aggregated_tags_pipeline.pkl'
with MODEL_PATH.open('rb') as file:
    artifact = pickle.load(file)

print('Версия:', artifact['version'])
for name, scaler in artifact['scalers'].items():
    print(name, 'min =', scaler.data_min_[0], 'max =', scaler.data_max_[0])

## Новые данные

- `TAG_JOIN_IND` — техническая колонка. Её удаляем.
- Пропуски в TAG считаем нулями.
- Договоры без заполненных TAG не участвуют в расчёте.
- Ниже выводим число исключённых договоров.

In [ ]:
part1 = pd.read_csv(DATA_CSV_PART1, encoding='cp1251', delimiter=',')
part2 = pd.read_csv(DATA_CSV_PART2, encoding='cp1251', delimiter=',')
data = pd.merge(part1, part2, on='POLICY_ZV', how='inner')
data.set_index('POLICY_ZV', inplace=True)

if 'TAG_JOIN_IND' in data.columns:
    data.drop(columns=['TAG_JOIN_IND'], inplace=True)

data['SUM'] = data.filter(like='TAG_').fillna(0).sum(axis=1)
rows_without_tags = int(data['SUM'].le(0).sum())
data = data[data['SUM'] > 0].copy()

print('Исключено договоров без TAG:', rows_without_tags)
print('Осталось строк:', len(data))

## Сырой скор

Для `auto_lover` и `shopping` умножаем каждый TAG на его вес и складываем. Для `alcohol` просто складываем значения TAG.

In [ ]:
raw_scores = pd.DataFrame(index=data.index)

for group_name in ('auto_lover', 'shopping'):
    selected_tags = artifact['feature_tags'][group_name]
    weights = pd.Series(artifact['weights'][group_name], dtype=float)
    X = data.reindex(columns=selected_tags, fill_value=0).fillna(0).astype(float)
    raw_scores[f'{group_name}_raw_score'] = (
        X.mul(weights, axis=1).sum(axis=1)
    )

alcohol_tags = artifact['feature_tags']['alcohol']
X_alcohol = data.reindex(columns=alcohol_tags, fill_value=0).fillna(0).astype(float)
raw_scores['alcohol_raw_score'] = X_alcohol.sum(axis=1)

raw_scores.head()

## MinMaxScaler

Берём scaler из `.pkl` и применяем только `transform`. Минимум и максимум на новых данных не пересчитываются. Благодаря `clip=True` значения ниже исходной границы становятся 0, а выше границы — 1.

In [ ]:
result = pd.DataFrame(index=data.index)

for group_name in ('auto_lover', 'shopping', 'alcohol'):
    raw_column = f'{group_name}_raw_score'
    result[f'{group_name}_agg_coef'] = (
        artifact['scalers'][group_name]
        .transform(raw_scores[[raw_column]])
        .ravel()
    )

result.head()